# Pneumonia Detection from Chest X-rays – End-to-End ML Pipeline
Binary image classification (**NORMAL** vs **PNEUMONIA**) on the Kaggle Chest X-Ray Images (Pneumonia) dataset (paediatric chest radiographs).
This notebook covers: data acquisition → preprocessing → model creation → training with optimisation → testing with 4+ metrics → prediction function → retraining demo.

## 0. Setup

In [ ]:
import sys, random, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, roc_auc_score,
                             accuracy_score, precision_score, recall_score, f1_score, log_loss)

ROOT = Path("..").resolve()          # notebook/ lives one level below the repo root
sys.path.insert(0, str(ROOT))

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
IMG_SIZE, BATCH = (224, 224), 32
TRAIN_DIR, TEST_DIR = ROOT / "data/train", ROOT / "data/test"
MODEL_PATH = ROOT / "models/pneumonia_cnn.h5"

## 1. Data acquisition
Needs a Kaggle token (`~/.kaggle/kaggle.json`). Run once from the repo root; it keeps the official split (`train`+`val` → `data/train`, `test` → `data/test`):
```bash
python scripts/download_data.py
```

In [ ]:
def list_images(root):
    paths, labels = [], []
    for idx, name in enumerate(CLASS_NAMES):
        for p in sorted((Path(root) / name).glob("*.jpeg")):
            paths.append(str(p)); labels.append(idx)
    return paths, labels

train_paths, train_labels = list_images(TRAIN_DIR)
test_paths, test_labels = list_images(TEST_DIR)
print(f"train: {len(train_paths)}  test: {len(test_paths)}")
print(pd.Series(train_labels).map(dict(enumerate(CLASS_NAMES))).value_counts())

## 2. Exploratory analysis – what story do the features tell?

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))  # sample X-rays per class
for r, cls in enumerate(CLASS_NAMES):
    sample = random.Random(SEED).sample([p for p, l in zip(train_paths, train_labels) if l == r], 6)
    for c, p in enumerate(sample):
        axes[r, c].imshow(plt.imread(p), cmap="gray"); axes[r, c].axis("off")
    axes[r, 0].set_title(cls, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
from PIL import Image
def image_features(path):
    img = Image.open(path).convert("RGB").resize(IMG_SIZE)
    rgb = np.asarray(img, dtype=np.float32); gray = rgb.mean(axis=2)
    return {"brightness": gray.mean(), "contrast": gray.std(), "bright_pixel_fraction": (gray > 200).mean()}

rng = random.Random(SEED)
rows = []
for i, cls in enumerate(CLASS_NAMES):
    for p in rng.sample([p for p, l in zip(train_paths, train_labels) if l == i], 500):
        rows.append({"class": cls, **image_features(p)})
feat = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, f in zip(axes, ["brightness", "contrast", "bright_pixel_fraction"]):
    for cls in CLASS_NAMES:
        ax.hist(feat.loc[feat["class"] == cls, f], bins=30, alpha=0.6, label=cls)
    ax.set_title(f); ax.legend()
plt.show()
feat.groupby("class").mean()

**Interpretation (fill in with your own numbers after running):**
1. *Brightness* – fluid and infection absorb X-rays, so pneumonia chests tend to look whiter overall.
2. *Contrast* – hazy opacities blur rib/lung boundaries and change the spread of grey levels.
3. *Bright-pixel fraction* – consolidation appears as dense white patches, so pneumonia images have more very white pixels.

The features overlap heavily, so simple thresholds are not enough – this motivates a CNN that learns texture and shape. Also note the class imbalance (about 3x more PNEUMONIA than NORMAL), handled with class weights.

## 3. Preprocessing
* Decode PNG → RGB, resize to 64×64 (same function at train and inference time)
* Stratified 85/15 train/validation split (test set kept untouched)
* Pixel rescaling 0–255 → 0–1 is the first layer **inside** the model
* Augmentation (random flips/rotation) is also inside the model and only active during training
* Class weights to handle the NORMAL/PNEUMONIA imbalance

In [ ]:
tr_p, va_p, tr_l, va_l = train_test_split(train_paths, train_labels, test_size=0.15,
                                          stratify=train_labels, random_state=SEED)

def decode_resize(raw):
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE); img.set_shape((*IMG_SIZE, 3))
    return tf.cast(img, tf.float32)

def make_dataset(paths, labels, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, np.asarray(labels, dtype=np.float32)))
    if training: ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(lambda p, y: (decode_resize(tf.io.read_file(p)), y), num_parallel_calls=tf.data.AUTOTUNE)
    if training:   # contrast/brightness jitter (in the data pipeline so the saved .h5 stays reloadable)
        def aug(img, y):
            img = tf.image.random_contrast(img, 0.8, 1.2); img = tf.image.random_brightness(img, 20.0)
            return tf.clip_by_value(img, 0.0, 255.0), y
        ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_dataset(tr_p, tr_l, True), make_dataset(va_p, va_l, False), make_dataset(test_paths, test_labels, False)
cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=np.array(tr_l))
class_weight = {0: float(cw[0]), 1: float(cw[1])}
print(len(tr_p), len(va_p), len(test_paths), class_weight)

## 4. Model creation
Transfer learning: an **ImageNet-pretrained MobileNetV2** backbone plus a small trainable head, trained in two phases (head only, then low-LR fine-tuning of the top backbone layers).
Optimisation techniques: **pretrained model, fine-tuning, data augmentation (flip/rotation/zoom in the model, contrast/brightness jitter in the data pipeline), dropout, L2 regularisation, Adam optimiser, early stopping, learning-rate scheduling, class weights.**

In [ ]:
def compile_model(m, lr):
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
              metrics=[keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.Precision(name="precision"),
                       keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")])

def build_model(lr=1e-3):
    l2 = regularizers.l2(1e-4)
    inp = keras.Input(shape=(*IMG_SIZE, 3))
    x = layers.RandomFlip("horizontal")(inp)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomZoom(0.1)(x)
    x = layers.Rescaling(1/127.5, offset=-1)(x)           # MobileNetV2 expects [-1, 1]
    base = keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
    base.trainable = False                                 # phase 1: head only
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=l2)(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)         # P(PNEUMONIA)
    m = keras.Model(inp, out, name="pneumonia_cnn")
    compile_model(m, lr)
    return m

def unfreeze_top(m, n_layers=60):
    base = next(l for l in m.layers if isinstance(l, keras.Model))
    base.trainable = True
    for layer in base.layers[:-n_layers]: layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization): layer.trainable = False

model = build_model()
model.summary()

## 5. Training

In [ ]:
def make_callbacks():
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)]

# phase 1: train the new head with the backbone frozen
h1 = model.fit(train_ds, validation_data=val_ds, epochs=5, class_weight=class_weight, callbacks=make_callbacks(), verbose=2)
# phase 2: fine-tune the top of the backbone at a low learning rate
unfreeze_top(model); compile_model(model, 1e-5)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight, callbacks=make_callbacks(), verbose=2)

merged = {k: h1.history[k] + h2.history[k] for k in h1.history}
history = type("History", (), {"history": merged})()      # same interface used by the cells below

In [ ]:
h = pd.DataFrame(history.history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
h[["loss", "val_loss"]].plot(ax=ax[0], title="Loss"); h[["accuracy", "val_accuracy"]].plot(ax=ax[1], title="Accuracy")
plt.show()

## 6. Testing – accuracy, loss, precision, recall, F1, ROC-AUC, confusion matrix
Precision/recall/F1 use **PNEUMONIA as the positive class**: a missed pneumonia case is the costly mistake.

In [ ]:
probs = model.predict(test_ds, verbose=0).ravel()
y_true = np.array(test_labels); y_pred = (probs >= 0.5).astype(int)

results = {
    "accuracy":  accuracy_score(y_true, y_pred),
    "loss":      log_loss(y_true, np.clip(probs, 1e-7, 1 - 1e-7)),
    "precision": precision_score(y_true, y_pred, pos_label=1),
    "recall":    recall_score(y_true, y_pred, pos_label=1),
    "f1":        f1_score(y_true, y_pred, pos_label=1),
    "roc_auc":   roc_auc_score(y_true, probs),
}
print(pd.Series(results).round(4))
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fpr, tpr, _ = roc_curve(y_true, probs)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(cm, cmap="Blues"); ax[0].set_xticks([0, 1], CLASS_NAMES); ax[0].set_yticks([0, 1], CLASS_NAMES)
for i in range(2):
    for j in range(2): ax[0].text(j, i, cm[i, j], ha="center", va="center")
ax[0].set_title("Confusion matrix"); ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual")
ax[1].plot(fpr, tpr, label=f"AUC={results['roc_auc']:.3f}"); ax[1].plot([0, 1], [0, 1], "--")
ax[1].set_title("ROC curve"); ax[1].legend(); plt.show()

## 7. Save the model (used by the API) and the metrics (shown in the UI)

In [ ]:
(ROOT / "models").mkdir(exist_ok=True)
model.save(MODEL_PATH)
(ROOT / "models/metrics.json").write_text(json.dumps({
    "kind": "initial", "trained_at": pd.Timestamp.utcnow().isoformat(), "n_train": len(tr_p), "n_val": len(va_p),
    "test": {**results, "n_samples": len(y_true), "confusion_matrix": cm.tolist()},
    "previous_test": None, "history": {k: [float(x) for x in v] for k, v in history.history.items()}}, indent=2))
print("saved", MODEL_PATH)

## 8. Prediction function

In [ ]:
def predict_image(path):
    model_ = keras.models.load_model(MODEL_PATH, compile=False)
    x = decode_resize(tf.io.read_file(str(path))).numpy()[None]
    p = float(model_(x, training=False).numpy().ravel()[0])
    return {"label": CLASS_NAMES[int(p >= 0.5)], "P(PNEUMONIA)": round(p, 4)}

for p, l in list(zip(test_paths, test_labels))[:5]:
    print(CLASS_NAMES[l], "->", predict_image(p))

## 9. Retraining
Retraining is implemented in `src/retrain.py` and triggered from the UI (or automatically after `RETRAIN_THRESHOLD`
uploads). It **loads the saved model above as a pre-trained model**, fine-tunes it at a low learning rate on the
uploaded images plus a replay sample of the original data, compares before/after on the test set and saves the new model.
Command-line equivalent after adding files under `data/uploads/<Class>/`:
```bash
python -m src.retrain
```